# Notebook 5 — Interpretation & Validation

**Project:** Robson Classification and C-Section Predictive System
**Stage:** Phase 4 — Explaining the model and proving it generalises
**Scripts covered (4):**

| Script | Role |
|--------|------|
| `step14_shap.py` | Global SHAP explanation — what drives the model |
| `step14b_shap_within.py` | Within-group SHAP — what actually discriminates inside a Robson group |
| `temporal_validation_2024.py` | Apply the 2023 classifier + model to 2024 data unchanged |
| `kappa_audit.py` | Manual verification tool for the classifier (prepared, not yet executed) |

---

### The three questions this notebook answers

1. **What does the model rely on?** (SHAP — step 14)
2. **Is the model telling us about biology or about practice?** (Within-group SHAP — step 14b)
3. **Does any of this hold on data the model has never seen from a different year?** (Temporal validation)

### The fundamental caveat

**SHAP explains the MODEL, not the medicine.** If the data contain a pattern driven by practice rather than physiology, SHAP faithfully reports that pattern as important — because it IS important to the model. Reading it as causal is the most common misuse of these plots.

---

## Step 14 — Global SHAP Explanation

**Script:** `step14_shap.py`

### Two sanity checks before reading anything else

| Feature | Expected | Why |
|---------|----------|-----|
| `RF_CESARN` | Near-zero importance | Training population is Groups 1, 2a, 3, 4a — no woman has a previous CS |
| `PRIORLIVE` | Top 1-3 | Parity separates Groups 1/2a from 3/4a, which sit at very different CS rates |

If `RF_CESARN` shows real influence → something leaked. If `PRIORLIVE` dominates → the model is working through a mechanism we can name (good sign).

### What the ranking found

**Anthropometric + demographic factors** (parity, birth interval, BMI, age, height, weight gain) account for ~77% of the model's decision-making. Actual medical comorbidities (hypertension, diabetes, eclampsia) combined contribute ~6%.

This means the model is driven primarily by body size and reproductive history, not by the clinical risk factors traditionally emphasised in obstetric practice.

### Outputs
- `shap_summary.png` — direction and magnitude per feature (beeswarm plot)
- `shap_importance.png` — mean absolute contribution, ranked (bar plot)
- `shap_importance.csv` — the same as numbers
- `shap_by_group.csv` — which features matter in which Robson group
- `shap_dependence_*.png` — how the top features behave across their range

In [ ]:
"""
=============================================================================
STEP 14 — SHAP
=============================================================================
    pip install shap matplotlib
    python step14_shap.py

WHAT SHAP DOES AND WHAT IT DOES NOT DO

SHAP explains THE MODEL. It says which inputs moved this model's output and
in which direction. It does not say what causes a caesarean. If the data
contain a pattern driven by practice rather than physiology, SHAP will
faithfully report that pattern as important, because it is important — to
the model. Reading it as a causal account is the most common way these
plots are misused, and the distinction belongs in the write-up.

TreeExplainer is used because it is exact for tree ensembles rather than an
approximation, and fast enough to run on a real sample.

TWO CHECKS WORTH MAKING BEFORE READING ANYTHING ELSE

  RF_CESARN should be almost inert. The training population is groups 1,
  2a, 3 and 4a, none of whom has a previous caesarean by definition. If the
  number of previous caesareans shows real influence, something has leaked.

  PRIORLIVE should be strong. The model cannot see onset of labour or the
  Robson group — both were excluded. Parity is the main thing separating
  groups 1 and 2a from 3 and 4a, and those pairs sit at very different
  caesarean rates. If parity dominates, the model is working through a
  mechanism we can name, which is a good sign rather than a surprising one.

OUTPUTS
    shap_summary.png            direction and magnitude per feature
    shap_importance.png         mean absolute contribution, ranked
    shap_importance.csv         the same, as numbers
    shap_by_group.csv           which features matter in which Robson group
    shap_dependence_*.png       how the top features behave across their range
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
import json

import xgboost as xgb
try:
    import shap
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError as e:
    raise SystemExit(f"Missing package: {e}\n  pip install shap matplotlib")

TARGET = "is_cesarean"
NARROW = ["1", "2a", "3", "4a"]
SAMPLE = 60_000          # enough for stable estimates, small enough to plot

wide = pd.read_parquet("train_wide.parquet")
FEATURES = [c for c in wide.columns
            if c not in [TARGET, "Robson_group", "Robson_group_sort_order", "_part"]]
for c in FEATURES:
    wide[c] = pd.to_numeric(wide[c], errors="coerce").astype("float32")

te = wide[wide["Robson_group"].isin(NARROW) & (wide["_part"] == "test")]
s = te.sample(n=min(SAMPLE, len(te)), random_state=42)
X = s[FEATURES]
print(f"Explaining {len(X):,} held-out births ({len(FEATURES)} features)\n")

bst = xgb.Booster(); bst.load_model("xgb_final.json")
expl = shap.TreeExplainer(bst)
sv = expl.shap_values(X)
print("SHAP values computed.\n")

# ---- ranked importance ----------------------------------------------------
imp = pd.DataFrame({
    "feature": FEATURES,
    "mean_abs_shap": np.abs(sv).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
imp["share_%"] = (100 * imp["mean_abs_shap"] / imp["mean_abs_shap"].sum()).round(2)

print("=" * 70)
print("RANKED CONTRIBUTION")
print("=" * 70)
print(imp.head(20).to_string(index=False))
imp.to_csv("shap_importance.csv", index=False)

# ---- the two sanity checks ------------------------------------------------
print("\n" + "=" * 70)
print("SANITY CHECKS")
print("=" * 70)
def rank_of(f):
    r = imp.index[imp.feature == f]
    return (int(r[0]) + 1, float(imp.loc[r[0], "share_%"])) if len(r) else (None, None)

r_cs, sh_cs = rank_of("RF_CESARN")
r_par, sh_par = rank_of("PRIORLIVE")
print(f"  RF_CESARN  rank {r_cs} of {len(imp)}, {sh_cs}% of total contribution")
print(f"    {'as expected — inert, since no woman here has a uterine scar' if sh_cs is not None and sh_cs < 2 else 'UNEXPECTED — investigate, this should be near zero'}")
print(f"  PRIORLIVE  rank {r_par} of {len(imp)}, {sh_par}% of total contribution")
print(f"    {'as expected — parity is the main thing separating the group pairs' if r_par and r_par <= 3 else 'lower than expected — worth understanding why'}")

# ---- plots ----------------------------------------------------------------
print("\nWriting plots ...")
plt.figure()
shap.summary_plot(sv, X, max_display=18, show=False)
plt.tight_layout(); plt.savefig("shap_summary.png", dpi=150); plt.close()

plt.figure()
shap.summary_plot(sv, X, plot_type="bar", max_display=18, show=False)
plt.tight_layout(); plt.savefig("shap_importance.png", dpi=150); plt.close()

for f in imp.head(4)["feature"]:
    plt.figure()
    shap.dependence_plot(f, sv, X, show=False)
    plt.tight_layout(); plt.savefig(f"shap_dependence_{f}.png", dpi=150); plt.close()
print("  shap_summary.png, shap_importance.png, 4 dependence plots")

# ---- per Robson group -----------------------------------------------------
# NOTE: This section originally used mean(|SHAP|) which inflated constant
# features (e.g. PRIORLIVE in Group 1). CORRECTED to std(SHAP) — see step14b.
# The old file is preserved as shap_by_group_OLD_mean_abs.csv.
print("\n" + "=" * 70)
print("WHICH FEATURES MATTER IN WHICH GROUP (std SHAP — corrected)")
print("=" * 70)
rows = []
for g in NARROW:
    m = (s["Robson_group"] == g).to_numpy()
    if m.sum() < 500:
        continue
    gi = pd.Series(sv[m].std(axis=0), index=FEATURES)
    gi = 100 * gi / gi.sum()
    for f, v in gi.sort_values(ascending=False).head(6).items():
        rows.append({"group": g, "feature": f, "share_%": round(float(v), 2)})
byg = pd.DataFrame(rows)
for g in NARROW:
    sub = byg[byg.group == g]
    if len(sub):
        parts = [f"{r['feature']} {r['share_%']}%" for _, r in sub.iterrows()]
        print(f"\n  Group {g}: " + " · ".join(parts))
byg.to_csv("shap_by_group.csv", index=False)

print("""

=====================================================================
HOW TO READ THIS IN THE WRITE-UP
=====================================================================
  A feature ranking high means the model relied on it. It does not mean
  that changing it would change the outcome. Two of the strongest signals
  in obstetric data — parity and gestational age — are also strongly
  associated with how clinicians manage a labour, so their importance
  here is a mixture of biology and practice that this data cannot
  separate.

  What the ranking CAN support is the negative claim the project rests
  on: if the measurable clinical picture accounted for the caesarean
  decision, a model given all of it would predict far better than 0.678 to
  0.705 within group. It does not. The residue is where practice lives,
  and that residue is what the Egypt comparison quantifies.
""")

json.dump({"top_features": imp.head(15).to_dict("records"),
           "n_explained": int(len(X))},
          open("shap_summary.json", "w"), indent=2)
print("Saved shap_importance.csv, shap_by_group.csv, shap_summary.json")
print("\nNext: step 15 — per-group performance is already in step 13; move to")
print("step 16, packaging the outputs for the dashboard.")

---

## Step 14b — What Actually Discriminates Inside a Robson Group

**Script:** `step14b_shap_within.py`

### The problem with Step 14's per-group table

Step 14 reported `PRIORLIVE` contributing 23% inside Group 1. But every woman in Group 1 is nulliparous — `PRIORLIVE` is zero for all of them. **A constant cannot distinguish one woman from another.**

### Why this happens

Mean |SHAP| measures departure from a baseline computed across the whole population (nulliparous AND multiparous). Being nulliparous pushes every Group 1 woman the same distance above that baseline. The contribution is large and **completely uninformative within the group** — it sets the level, it does not separate anybody.

**This also explains the gap between headline ROC-AUC (0.7991) and within-group ROC-AUC (0.678–0.705).** The two largest contributors (`PRIORLIVE` + `ILLB_R`, ~40% combined) are near-constant inside a group. They separate GROUPS from each other and do almost nothing inside them.

### The fix

Within each group, rank by **variance** (std) of SHAP values instead of mean absolute value. A constant has zero variance and disappears. What remains is what genuinely tells one woman apart from another with the same Robson classification.

### The key finding (std SHAP — corrected)

| Group | Maternal body + age | Medical risk factors |
|-------|--------------------|--------------------|
| 1 | 60.6% | 14.3% |
| 2a | 57.9% | 21.2% |
| 3 | 45.4% | 12.8% |
| 4a | 44.8% | 17.7% |

**Among women with no uterine scar, at term, with a single cephalic fetus, documented medical indications explain very little of who ends up with a caesarean.** The model separates women almost entirely on maternal size and age.

This is the same negative claim the project rests on, arriving from a different direction.

In [ ]:
"""
=============================================================================
STEP 14b — WHAT ACTUALLY DISCRIMINATES INSIDE A ROBSON GROUP
=============================================================================
    python step14b_shap_within.py

THE PROBLEM WITH THE STEP 14 TABLE

Step 14 reported PRIORLIVE contributing 23% inside Robson group 1. Every
woman in group 1 is nulliparous — PRIORLIVE is zero for all of them. A
constant cannot distinguish one woman from another, so that 23% is not
telling us what it appears to.

The reason is how mean |SHAP| works. SHAP measures departure from a
baseline computed across the whole population, nulliparous and multiparous
together. Being nulliparous pushes every woman in group 1 the same distance
above that baseline. The contribution is large and completely uninformative
within the group: it sets the level, it does not separate anybody.

That also explains the gap between the headline ROC-AUC of 0.7991 and the
within-group figures of 0.678 to 0.705. The two largest contributors,
PRIORLIVE and ILLB_R at about 40% combined, are near-constant inside a
group. They separate the groups from each other and do almost nothing
inside them.

THE FIX

Within each group, rank features by the VARIANCE of their SHAP values
rather than the mean absolute value. A feature that is constant has zero
variance and disappears. What remains is what genuinely tells one woman
apart from another with the same Robson classification — which is the only
thing a clinician standing in front of a patient in that group can use.

Both columns are reported side by side, because the difference between them
is itself the finding.
=============================================================================
"""

import pandas as pd
import numpy as np
import xgboost as xgb
import shap

TARGET = "is_cesarean"
NARROW = ["1", "2a", "3", "4a"]
SAMPLE = 60_000

wide = pd.read_parquet("train_wide.parquet")
FEATURES = [c for c in wide.columns
            if c not in [TARGET, "Robson_group", "Robson_group_sort_order", "_part"]]
for c in FEATURES:
    wide[c] = pd.to_numeric(wide[c], errors="coerce").astype("float32")

te = wide[wide["Robson_group"].isin(NARROW) & (wide["_part"] == "test")]
s = te.sample(n=min(SAMPLE, len(te)), random_state=42).reset_index(drop=True)
X = s[FEATURES]

bst = xgb.Booster(); bst.load_model("xgb_final.json")
sv = shap.TreeExplainer(bst).shap_values(X)
print(f"Explained {len(X):,} births\n")

# ---- overall, for reference ----------------------------------------------
overall = pd.DataFrame({
    "feature": FEATURES,
    "mean_abs": np.abs(sv).mean(axis=0),
    "sd": sv.std(axis=0),
})
overall["mean_abs_%"] = (100*overall.mean_abs/overall.mean_abs.sum()).round(2)
overall["sd_%"] = (100*overall.sd/overall.sd.sum()).round(2)

print("=" * 76)
print("WHOLE POPULATION — the two rankings already differ")
print("=" * 76)
o = overall.sort_values("mean_abs", ascending=False).head(12)
print(o[["feature", "mean_abs_%", "sd_%"]].to_string(index=False))

# ---- within group ---------------------------------------------------------
print("\n" + "=" * 76)
print("WITHIN EACH ROBSON GROUP")
print("=" * 76)
print("""
  mean_abs_%  how far this feature pushes predictions from the global
              baseline — includes constants that move everyone equally
  sd_%        how much this feature VARIES between women in the group —
              a constant scores zero here
""")

rows = []
for g in NARROW:
    m = (s["Robson_group"] == g).to_numpy()
    if m.sum() < 500:
        continue
    sub_sv = sv[m]
    t = pd.DataFrame({
        "feature": FEATURES,
        "mean_abs": np.abs(sub_sv).mean(axis=0),
        "sd": sub_sv.std(axis=0),
        "value_sd": X[m].std(axis=0).to_numpy(),
    })
    t["mean_abs_%"] = (100*t.mean_abs/t.mean_abs.sum()).round(2)
    t["sd_%"] = (100*t.sd/t.sd.sum()).round(2)
    t["group"] = g
    t = t.sort_values("sd", ascending=False)
    rows.append(t)

    print(f"\n  ── GROUP {g}  (n={int(m.sum()):,}, "
          f"caesarean {100*s.loc[m, TARGET].mean():.2f}%) " + "─"*28)
    print(t.head(8)[["feature", "sd_%", "mean_abs_%"]].to_string(index=False))

    # features that set the level but do not discriminate
    dead = t[(t["sd_%"] < 0.5) & (t["mean_abs_%"] > 3)]
    if len(dead):
        print("    level-setting only (large contribution, no variation here):")
        for _, r in dead.iterrows():
            print(f"      {r['feature']:<18} mean_abs {r['mean_abs_%']:>5.2f}%   "
                  f"sd {r['sd_%']:>5.2f}%")

allg = pd.concat(rows, ignore_index=True)
allg[["group", "feature", "sd_%", "mean_abs_%"]].to_csv(
    "shap_within_group.csv", index=False)

# ---- the summary that goes in the write-up -------------------------------
print("\n" + "=" * 76)
print("WHAT THIS MEANS")
print("=" * 76)

top = (allg.sort_values(["group", "sd_%"], ascending=[True, False])
           .groupby("group").head(4))
print("\n  Ranked by what actually varies within the group:\n")
for g in NARROW:
    sub = top[top.group == g]
    if len(sub):
        print(f"    Group {g}: " + " · ".join(
            f"{r['feature']} {r['sd_%']}%" for _, r in sub.iterrows()))

body = ["BMI", "M_Ht_In", "PWgt_R", "WTGAIN", "MAGER"]
risk = ["RF_PDIAB", "RF_GDIAB", "RF_PHYPE", "RF_GHYPE", "RF_EHYPE",
        "RF_PPTERM", "RF_INFTR", "RF_ARTEC"]
b = allg[allg.feature.isin(body)].groupby("group")["sd_%"].sum()
r = allg[allg.feature.isin(risk)].groupby("group")["sd_%"].sum()
print(f"\n  {'group':<8} {'maternal body + age':>22} {'medical risk factors':>24}")
for g in NARROW:
    if g in b.index:
        print(f"  {g:<8} {b[g]:>21.1f}% {r.get(g, 0):>23.1f}%")

print("""
  Read those two columns against each other.

  Once the Robson group is fixed — which is what a clinician actually knows
  when a woman is in front of them — the model separates one woman from
  another almost entirely on maternal size and age. The recorded medical
  risk factors, the conditions that would ordinarily justify a caesarean,
  account for a small fraction of the discrimination.

  That is not an argument that those conditions do not matter clinically.
  It is a statement about what is in this data: among women with no
  uterine scar, at term, with a single cephalic fetus, documented medical
  indications explain very little of who ends up with a caesarean and who
  does not.

  This is the same negative claim the project rests on, arriving from a
  different direction.
""")
print("Saved shap_within_group.csv")

---

## Temporal Validation — 2024

**Script:** `temporal_validation_2024.py`

### Why this matters more than anything else added so far

Every figure in the project comes from one year. The obvious question — does this hold next year? — has not been answered. Most student projects never answer it. It takes one run, because the model and classifier already exist.

### Two separate things tested (can fail independently)

**1. The classifier:** Run the same Robson rules on 2024 data and compare group sizes and CS rates. Real shifts are expected; large ones would mean a coding change we missed.

**2. The model:** The 2023-trained XGBoost applied to 2024 without refitting.
- **Discrimination** usually survives a year
- **Calibration** is the fragile part — and calibration is what the Egypt calculation depends on (it sums probabilities)

### Field positions

All 48 positions were verified against the 2024 User Guide. None moved. The same colspecs are reused unchanged.

### Honesty note

This is a single run and its result is reported whatever it says. If calibration degrades, that goes in the limitations as a finding about how long a fitted model stays usable.

In [ ]:
"""
=============================================================================
TEMPORAL VALIDATION — 2024
=============================================================================
    python temporal_validation_2024.py Nat2024PublicUS.txt

WHY THIS MATTERS MORE THAN ANYTHING ELSE ADDED SO FAR

Every figure in the project comes from one year. The obvious question — does
this hold next year, or was it fitted to the peculiarities of 2023? — has
not been answered. Most student projects never answer it. It takes one run,
because the model and the classifier already exist and nothing needs
retraining.

Two separate things are being tested, and they can fail independently:

  1. THE CLASSIFIER. Run the same rules on 2024 and compare group sizes and
     group caesarean rates. This is a stability check on the data and on the
     derivation, especially onset of labour. Real shifts are expected and
     interesting; large ones would mean a coding change we missed.

  2. THE MODEL. The 2023-trained model is applied to 2024 without any
     refitting. Discrimination usually survives a year. CALIBRATION is the
     fragile part, and calibration is what this project depends on: the
     Egypt calculation adds up probabilities. A model whose probabilities
     drift a year later is a model whose expected rates cannot be trusted.

FIELD POSITIONS
All 48 positions used here were checked against the 2024 User Guide before
this was written. None moved. The same colspecs are reused unchanged, which
is why this can run at all.

HONESTY NOTE
This is a single run and its result is reported whatever it says. If
calibration degrades on 2024, that goes in the limitations as a finding
about how long a fitted model stays usable — which is worth knowing.
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
import sys, json, gc

sys.path.insert(0, str(Path(__file__).parent))
from robson_classifier import classify, report_table
from cdc_robson_pipeline import (COLSPECS as BASE_COLSPECS,
                                 COLNAMES as BASE_COLNAMES,
                                 add_is_cesarean, add_practice_vars,
                                 add_near_miss_flag)

# The six columns added in stage 4 live in stage4_setting_and_wider_adjustment.py,
# not in the original pipeline. The 2023 run built them in a second pass over the
# raw file; here they are read in the same pass. Positions verified against the
# 2024 User Guide — identical to 2023.
EXTRA_COLSPECS = [
    (31, 32),     # BFACIL      pos 32
    (119, 120),   # DMAR        pos 120
    (123, 124),   # MEDUC       pos 124
    (226, 227),   # PRECARE5    pos 227
    (268, 269),   # CIG_REC     pos 269
    (317, 318),   # RF_PPTERM   pos 318
]
EXTRA_COLNAMES = ["BFACIL", "DMAR", "MEDUC", "PRECARE5", "CIG_REC", "RF_PPTERM"]

COLSPECS = BASE_COLSPECS + [c for c, n in zip(EXTRA_COLSPECS, EXTRA_COLNAMES)
                            if n not in BASE_COLNAMES]
COLNAMES = BASE_COLNAMES + [n for n in EXTRA_COLNAMES if n not in BASE_COLNAMES]

RAW = sys.argv[1] if len(sys.argv) > 1 else "Nat2024PublicUS.txt"
CACHE = "natality_2024_classified.parquet"
TARGET = "is_cesarean"
NARROW = ["1", "2a", "3", "4a"]

# =============================================================================
if Path(CACHE).exists():
    print(f"Loading cached {CACHE} ...")
    d24 = pd.read_parquet(CACHE)
else:
    if not Path(RAW).exists():
        raise SystemExit(f"{RAW} not found.")
    print(f"Reading {RAW} ...")
    print("  Field positions were verified identical to 2023 against the")
    print("  2024 User Guide — all 48, none moved.\n")
    frames = []
    for i, ch in enumerate(pd.read_fwf(RAW, colspecs=COLSPECS, names=COLNAMES,
                                       dtype=str, chunksize=500_000)):
        frames.append(ch)
        if (i + 1) % 3 == 0:
            print(f"  {sum(len(f) for f in frames):,} rows")
    d24 = pd.concat(frames, ignore_index=True)
    del frames; gc.collect()
    print(f"  {len(d24):,} births loaded\n")

    d24 = add_is_cesarean(d24)
    d24 = add_practice_vars(d24)
    d24 = add_near_miss_flag(d24)
    print("Classifying ...")
    d24 = classify(d24, subgroups=True)
    d24.to_parquet(CACHE, index=False)
    print(f"Cached to {CACHE}\n")

c24 = d24[d24["Classification_status"] == "Classified"]
print("=" * 76)
print("PART 1 — DOES THE CLASSIFIER REPRODUCE ITSELF ON A NEW YEAR?")
print("=" * 76)
print(f"  2024: {len(d24):,} births, {len(c24):,} classified "
      f"({100*(len(d24)-len(c24))/len(d24):.2f}% unclassifiable)")
print(f"  2023: 3,605,081 births, 3,585,385 classified (0.55% unclassifiable)")
print(f"  Overall caesarean rate 2024: {100*c24[TARGET].mean():.2f}%   (2023: 32.35%)\n")

R23 = {"1":(16.43,13.00),"2a":(15.01,29.30),"2b":(2.48,100.00),"3":(21.98,2.52),
       "4a":(14.44,5.89),"4b":(2.23,100.00),"5.1":(8.77,78.27),"5.2":(3.75,95.37),
       "6":(1.72,96.24),"7":(1.73,93.78),"8":(3.16,75.07),"9":(0.83,46.52),
       "10":(7.46,43.17)}

t = (c24.groupby(["Robson_group_sort_order","Robson_group"], observed=True)
        .agg(n=(TARGET,"size"), cs=(TARGET,"mean")).reset_index()
        .sort_values("Robson_group_sort_order"))
t["size_24"] = (100*t["n"]/len(c24)).round(2)
t["rate_24"] = (100*t["cs"]).round(2)

print(f"{'group':>6} {'size 23':>9} {'size 24':>9} {'Δ':>7}   "
      f"{'rate 23':>9} {'rate 24':>9} {'Δ':>7}")
print("-"*66)
rows=[]
for _, r in t.iterrows():
    g = r["Robson_group"]
    if g not in R23: continue
    s23, c23_ = R23[g]
    ds, dc = r["size_24"]-s23, r["rate_24"]-c23_
    flag = "  <-- shift" if abs(dc) > 3 or abs(ds) > 2 else ""
    print(f"{g:>6} {s23:>8.2f}% {r['size_24']:>8.2f}% {ds:>+7.2f}   "
          f"{c23_:>8.2f}% {r['rate_24']:>8.2f}% {dc:>+7.2f}{flag}")
    rows.append({"group":g,"size_2023":s23,"size_2024":float(r["size_24"]),
                 "rate_2023":c23_,"rate_2024":float(r["rate_24"]),
                 "size_delta":round(float(ds),2),"rate_delta":round(float(dc),2)})

ntsv = c24[c24["Robson_group"].isin(["1","2a","2b"])]
print(f"\n  NTSV 2024: {100*ntsv[TARGET].mean():.2f}%   (2023: 26.56%)")
print("""
  Groups 2b and 4b must stay at exactly 100% — they are defined that way.
  Anything else there would mean the onset-of-labour derivation behaves
  differently on this year's file.
""")

# =============================================================================
print("=" * 76)
print("PART 2 — DOES THE 2023 MODEL STILL WORK IN 2024?")
print("=" * 76)
print("  No refitting. The model trained on 2023 is applied unchanged.\n")

try:
    import xgboost as xgb
    from sklearn.metrics import (roc_auc_score, average_precision_score,
                                 brier_score_loss)
except ImportError as e:
    raise SystemExit(f"Missing package: {e}")

if not Path("xgb_final.json").exists():
    raise SystemExit("xgb_final.json not found — run step11b_calibration.py first.")

FEATURES = pd.read_csv("model_feature_order.csv")["feature"].tolist()

ENGINEERED = {"ILLB_first_birth", "ME_PRES_cephalic", "ME_PRES_breech", "ME_PRES_other"}
need_raw = [f for f in FEATURES if f not in ENGINEERED]
absent = [f for f in need_raw if f not in c24.columns]
if absent:
    raise SystemExit(
        f"These model features were not read from the raw file: {absent}\n"
        "Delete natality_2024_classified.parquet and re-run, so the cache is "
        "rebuilt with the full column set.")

# rebuild the engineered features exactly as steps 5, 6 did
n24 = c24[c24["Robson_group"].isin(NARROW)].copy()
SENT = {"M_Ht_In":[99],"PWgt_R":[999],"WTGAIN":[99],"BMI":[99.9],"MEDUC":[9],
        "DMAR":[9],"PRIORLIVE":[99],"PRIORDEAD":[99],"RF_CESARN":[99],
        "ILLB_R":[888,999],"OEGest_Comb":[99],"DPLURAL":[9]}
YN = ["CIG_REC","RF_PPTERM","RF_PDIAB","RF_GDIAB","RF_PHYPE","RF_GHYPE",
      "RF_EHYPE","RF_INFTR","RF_ARTEC"]

for c in YN:
    s = n24[c].astype(str).str.strip().str.upper()
    v = pd.Series(np.nan, index=n24.index)
    v[s=="Y"]=1.0; v[s=="N"]=0.0
    n24[c]=v
for c, codes in SENT.items():
    x = pd.to_numeric(n24[c], errors="coerce")
    n24[c] = x.where(~x.isin(codes))
pc = n24["PRECARE5"].astype(str).str.strip()
n24["PRECARE5"] = pd.to_numeric(pc.where(pc!="5"), errors="coerce")
# conditional recode, same rule as step 6
n24.loc[n24["RF_INFTR"]==0.0, "RF_ARTEC"] = 0.0
n24["ILLB_first_birth"] = (pd.to_numeric(n24["PRIORLIVE"],errors="coerce")==0).astype(float)
pres = n24["ME_PRES"].astype(str).str.strip()
for code,name in {"1":"cephalic","2":"breech","3":"other"}.items():
    n24[f"ME_PRES_{name}"] = (pres==code).astype(float)
    n24.loc[pres=="9", f"ME_PRES_{name}"] = np.nan

X = n24[FEATURES].apply(pd.to_numeric, errors="coerce").astype("float32")
y = n24[TARGET].to_numpy().astype(int)

bst = xgb.Booster(); bst.load_model("xgb_final.json")
p = bst.predict(xgb.DMatrix(X))

T23 = {"roc_auc":0.7991,"auprc":0.3542,"brier":0.0886,"mean_pred":0.1169,
       "observed":0.1172,"cal_mae":0.11}
auc, ap, br = roc_auc_score(y,p), average_precision_score(y,p), brier_score_loss(y,p)

print(f"  n = {len(y):,} births in groups 1, 2a, 3 and 4a\n")
print(f"{'metric':<24} {'2023 test':>11} {'2024':>11} {'change':>10}")
print("-"*60)
for k,v23,v24 in [("ROC-AUC",T23['roc_auc'],auc),("AUPRC",T23['auprc'],ap),
                  ("Brier",T23['brier'],br),
                  ("mean predicted",T23['mean_pred'],p.mean()),
                  ("observed rate",T23['observed'],y.mean())]:
    print(f"{k:<24} {v23:>11.4f} {v24:>11.4f} {v24-v23:>+10.4f}")

q = pd.qcut(p, 10, labels=False, duplicates="drop")
cal = pd.DataFrame({"b":q,"pred":p,"obs":y}).groupby("b").agg(
    n=("obs","size"), predicted=("pred","mean"), observed=("obs","mean"))
cal["predicted_%"]=(100*cal.predicted).round(2)
cal["observed_%"]=(100*cal.observed).round(2)
cal["gap_pp"]=(cal["predicted_%"]-cal["observed_%"]).round(2)
mae = cal["gap_pp"].abs().mean()

print(f"\n  Calibration on 2024, by risk decile")
print(cal[["n","predicted_%","observed_%","gap_pp"]].to_string())
print(f"\n  mean absolute calibration error: {mae:.2f} pp   (2023 test: {T23['cal_mae']:.2f} pp)")

print("\n" + "=" * 76)
print("VERDICT")
print("=" * 76)
disc_ok = abs(auc - T23["roc_auc"]) < 0.02
cal_ok  = mae < 1.0
print(f"""
  Discrimination {'HOLDS' if disc_ok else 'DEGRADED'} — ROC-AUC moved {auc-T23['roc_auc']:+.4f}.
  Calibration    {'HOLDS' if cal_ok else 'DEGRADED'} — mean error {mae:.2f} pp against {T23['cal_mae']:.2f} pp.
""")
if disc_ok and cal_ok:
    print("""  The model transfers to a year it was never shown, without refitting.
  That is the answer to the first question any reviewer asks, and most
  student projects cannot answer it. Calibration holding matters more than
  discrimination here, because the Egypt estimate is built from summed
  probabilities rather than from rankings.
""")
else:
    print("""  Report this as it stands. A model that discriminates but drifts in
  calibration is still usable for ranking and NOT usable for estimating
  expected rates — which would change what this model may be used for,
  and belongs in the model card rather than being quietly smoothed over.
""")

json.dump({"year":2024,"n_classified":int(len(c24)),
           "overall_cs_2024":float(c24[TARGET].mean()),
           "ntsv_2024":float(ntsv[TARGET].mean()),
           "groups":rows,
           "model":{"roc_auc":float(auc),"auprc":float(ap),"brier":float(br),
                    "mean_predicted":float(p.mean()),"observed":float(y.mean()),
                    "calibration_mae_pp":float(mae)},
           "comparison_2023":T23},
          open("temporal_validation_2024.json","w"), indent=2)
pd.DataFrame(rows).to_csv("robson_2023_vs_2024.csv", index=False)
print("Saved temporal_validation_2024.json and robson_2023_vs_2024.csv")

---

## Kappa Audit — Prepared, Not Yet Executed

**Script:** `kappa_audit.py`

### What this tool does

1. **`python kappa_audit.py`** — Draws 50 records (40 random + 10 edge cases from Groups 6, 7, 8, 9, Unclassifiable). Produces a blinded worksheet with no algorithmic output, plus a separate answer key.

2. **You classify each record by hand** using the WHO flowchart (included as a guide file).

3. **`python kappa_audit.py --score`** — Compares your classification against the algorithm's and computes Cohen's Kappa on the random sample only (edge cases reported separately, excluded from the coefficient).

### Why it is designed this way

- **Blinding:** The worksheet contains no algorithm output. The answer key is separate. Opening it early invalidates the audit.
- **Two-part sampling:** Random sample → the coefficient means something. Edge cases → catches bugs but would deflate the coefficient if included.
- **The value is NOT the coefficient** (it should be 1.0 for deterministic code). The value is the **disagreements** — each one is either a bug or an ambiguity in how WHO definitions meet this data source.

### Current status

The tool is ready. The classifier is validated by:
- 21 unit tests (all PASS) ✓
- External rate matching (32.35% vs 32.3%) ✓
- Structural invariant checks ✓
- Temporal stability (2024 data) ✓

Manual chart review is a future enhancement that would add a published inter-rater reliability metric.

In [ ]:
"""
=============================================================================
COHEN'S KAPPA AUDIT — manual verification of the Robson classifier
=============================================================================
    python kappa_audit.py            # build the blinded worksheet
    python kappa_audit.py --score    # score it after you have filled it in

WHY THIS IS WORTH DOING, AND WHAT IT IS NOT

The classifier is deterministic if/else code. A clinician working from the
same variables should reach the same group every time, so kappa = 1 is the
expected result, not a surprising one. A published audit of a comparable
rule-based classifier (IJRCOG 2024) reported exactly that.

The value therefore is not the coefficient. It is the disagreements. A
single mismatch means one of two things, and both are worth finding:

  - a derivation bug, most likely in onset of labour, which is the only
    Robson criterion not recorded directly in this data source; or
  - a genuine ambiguity in the WHO definitions as they meet this data,
    which belongs in the limitations section rather than being silently
    resolved by whichever reading the code happened to take.

BLINDING
The worksheet contains no algorithmic output. The answer key is written to
a separate file. Do not open it until the worksheet is complete, or the
audit is worthless.

SAMPLING
50 records in two parts, kept separate on purpose:
  - 40 drawn at random. Kappa is computed on these, because they are
    representative and the coefficient means something.
  - 10 drawn purposively from the rare and awkward groups (6, 7, 8, 9,
    unclassifiable). These are reported as an edge-case check and are
    EXCLUDED from the kappa, because hand-picking hard cases would
    deflate a coefficient that is supposed to describe routine work.
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
import sys

WORKSHEET = "kappa_audit_worksheet.csv"
ANSWERKEY = "kappa_audit_answerkey.csv"
GUIDE     = "kappa_audit_guide.txt"

PRES = {"1": "Cephalic", "2": "Breech", "3": "Other", "9": "Unknown"}
ROUT = {"1": "Spontaneous vaginal", "2": "Forceps", "3": "Vacuum",
        "4": "Caesarean", "9": "Unknown"}
YN   = {"Y": "Yes", "N": "No", "U": "Unknown"}
PLUR = {"1": "Singleton", "2": "Twin", "3": "Triplet", "4": "Quadruplet",
        "5": "Quintuplet or higher"}


def decode(v, d):
    return d.get(str(v).strip(), f"[{v}]")


def build():
    src = next((f for f in ["natality_2023_extended.parquet",
                            "natality_2023_filtered.parquet"]
                if Path(f).exists()), None)
    if src is None:
        raise SystemExit("No parquet cache found. Run robson_analysis.py first.")

    print(f"Loading {src} ...")
    df = pd.read_parquet(src)
    rng = np.random.default_rng(2024)

    # 40 representative
    main = df.sample(n=40, random_state=2024)
    main = main.assign(part="random")

    # 10 edge cases, spread across the awkward groups
    edge_pool, want = [], ["6", "7", "8", "9", "Unclassifiable"]
    for g in want:
        sub = df[df["Robson_group"] == g]
        if len(sub):
            edge_pool.append(sub.sample(n=min(2, len(sub)), random_state=2024))
    edge = pd.concat(edge_pool).assign(part="edge") if edge_pool else main.iloc[:0]

    s = pd.concat([main, edge], ignore_index=True)
    s = s.sample(frac=1, random_state=7).reset_index(drop=True)   # shuffle
    s["audit_id"] = [f"A{i+1:03d}" for i in range(len(s))]

    ga = pd.to_numeric(s["OEGest_Comb"], errors="coerce")
    ga = ga.where(~ga.isin([99, 999]))

    def count(col):
        x = pd.to_numeric(s[col], errors="coerce")
        return x.where(~x.isin([99, 999])).apply(
            lambda v: "unknown" if pd.isna(v) else str(int(v)))

    ws = pd.DataFrame({
        "audit_id": s["audit_id"],
        "previous_live_births":  count("PRIORLIVE"),
        "previous_stillbirths":  count("PRIORDEAD"),
        "previous_caesarean":    s["RF_CESAR"].apply(lambda v: decode(v, YN)),
        "number_previous_caesareans": count("RF_CESARN"),
        "labour_induced":        s["LD_INDL"].apply(lambda v: decode(v, YN)),
        "trial_of_labour_attempted": s["ME_TRIAL"].apply(
            lambda v: decode(v, YN) + "  (recorded for caesareans only)"),
        "route_of_delivery":     s["ME_ROUT"].apply(lambda v: decode(v, ROUT)),
        "fetal_presentation":    s["ME_PRES"].apply(lambda v: decode(v, PRES)),
        "plurality":             s["DPLURAL"].apply(lambda v: decode(v, PLUR)),
        "gestational_age_weeks": ga.apply(
            lambda v: "unknown" if pd.isna(v) else str(int(v))),
        "YOUR_GROUP": "",
        "YOUR_NOTES": "",
    })
    ws.to_csv(WORKSHEET, index=False)

    s[["audit_id", "Robson_group", "Classification_status",
       "Missing_variable", "part"]].rename(
        columns={"Robson_group": "algorithm_group"}).to_csv(ANSWERKEY, index=False)

    Path(GUIDE).write_text(GUIDE_TEXT, encoding="utf-8")

    print(f"\nWritten:")
    print(f"  {WORKSHEET}   <- fill in the YOUR_GROUP column")
    print(f"  {GUIDE}       <- the WHO flowchart, in order")
    print(f"  {ANSWERKEY}   <- DO NOT OPEN until you are finished")
    print(f"\n{len(ws)} records: {(s['part']=='random').sum()} random, "
          f"{(s['part']=='edge').sum()} edge cases.")


def kappa(a, b):
    """Cohen's kappa for nominal categories, computed from first principles."""
    cats = sorted(set(a) | set(b))
    n = len(a)
    idx = {c: i for i, c in enumerate(cats)}
    m = np.zeros((len(cats), len(cats)))
    for x, y in zip(a, b):
        m[idx[x], idx[y]] += 1
    po = np.trace(m) / n
    pe = sum(m[i, :].sum() * m[:, i].sum() for i in range(len(cats))) / (n * n)
    k = (po - pe) / (1 - pe) if pe != 1 else 1.0
    return k, po, pe, pd.DataFrame(m, index=cats, columns=cats).astype(int)


def score():
    ws = pd.read_csv(WORKSHEET, dtype=str)
    key = pd.read_csv(ANSWERKEY, dtype=str)
    d = ws.merge(key, on="audit_id")

    d["YOUR_GROUP"] = d["YOUR_GROUP"].fillna("").str.strip()
    blank = (d["YOUR_GROUP"] == "").sum()
    if blank:
        print(f"WARNING: {blank} rows have no answer and are excluded.\n")
    d = d[d["YOUR_GROUP"] != ""].copy()
    if not len(d):
        raise SystemExit("No answers found. Fill in the YOUR_GROUP column first.")

    d["YOUR_GROUP"] = d["YOUR_GROUP"].str.replace(" ", "").str.capitalize()
    d.loc[d["YOUR_GROUP"].str.lower().str.startswith("unclass"), "YOUR_GROUP"] = "Unclassifiable"

    print("=" * 66)
    print("PRIMARY RESULT — random sample only")
    print("=" * 66)
    r = d[d["part"] == "random"]
    if len(r):
        k, po, pe, cm = kappa(list(r["YOUR_GROUP"]), list(r["algorithm_group"]))
        print(f"  n                    : {len(r)}")
        print(f"  Observed agreement   : {100*po:.1f}%")
        print(f"  Expected by chance   : {100*pe:.1f}%")
        print(f"  Cohen's kappa        : {k:.3f}")
        print("""
  Landis & Koch: above 0.81 is almost perfect agreement. Because the
  classifier is deterministic, anything materially below 1.0 points at a
  derivation problem rather than at ordinary rater variability.
""")

    print("=" * 66)
    print("EDGE CASES — reported separately, excluded from kappa")
    print("=" * 66)
    e = d[d["part"] == "edge"]
    if len(e):
        agree = (e["YOUR_GROUP"] == e["algorithm_group"]).sum()
        print(f"  {agree} of {len(e)} agree")

    print("\n" + "=" * 66)
    print("DISAGREEMENTS — the actual point of the exercise")
    print("=" * 66)
    bad = d[d["YOUR_GROUP"] != d["algorithm_group"]]
    if not len(bad):
        print("  None. The implementation reproduces manual classification exactly.")
        print("  Report this as complete agreement and move on.")
    else:
        for _, row in bad.iterrows():
            print(f"\n  {row['audit_id']} ({row['part']})")
            print(f"    you: {row['YOUR_GROUP']:<16} algorithm: {row['algorithm_group']}")
            print(f"    parity {row['previous_live_births']}+{row['previous_stillbirths']}"
                  f" | prev CS {row['previous_caesarean']}"
                  f" | induced {row['labour_induced']}"
                  f" | trial {row['trial_of_labour_attempted'][:3]}"
                  f" | route {row['route_of_delivery']}")
            print(f"    {row['fetal_presentation']} | {row['plurality']}"
                  f" | {row['gestational_age_weeks']} weeks")
            if str(row.get("YOUR_NOTES", "")).strip() not in ("", "nan"):
                print(f"    your note: {row['YOUR_NOTES']}")
        print("""
  Work through each one. Every disagreement resolves to either a bug in the
  code or an ambiguity in the definitions as they meet this data source.
  The first gets fixed; the second gets written into the limitations.
""")
        bad.to_csv("kappa_audit_disagreements.csv", index=False)
        print("  Saved kappa_audit_disagreements.csv")


GUIDE_TEXT = """
ROBSON CLASSIFICATION — WORKING SHEET
=====================================

Classify each record in kappa_audit_worksheet.csv WITHOUT looking at the
answer key. Write the group in the YOUR_GROUP column. Use the subgroup
labels where they apply: 2a, 2b, 4a, 4b, 5.1, 5.2. Write "Unclassifiable"
if a criterion needed for the decision is unknown.

DERIVED DEFINITIONS
  Parity        Nulliparous means no previous birth at 22 weeks or later,
                live or stillborn. Add previous live births and previous
                stillbirths; zero means nulliparous.
  Term          37 completed weeks or more.
  Onset         Induced if labour was induced.
                Otherwise spontaneous if the birth was vaginal, or if a
                caesarean followed a recorded trial of labour.
                Otherwise pre-labour caesarean.
                Note: trial of labour is recorded for caesareans only. A
                vaginal birth had labour by definition.

WORK THROUGH IN THIS ORDER. Stop at the first match.

  1.  Two or more fetuses ....................................... GROUP 8
  2.  Singleton, transverse or oblique lie ...................... GROUP 9
  3.  Singleton breech, nulliparous ............................. GROUP 6
  4.  Singleton breech, multiparous ............................. GROUP 7
  5.  Singleton cephalic, under 37 weeks ........................ GROUP 10
  6.  Singleton cephalic, term, one previous caesarean ......... GROUP 5.1
  7.  Singleton cephalic, term, two or more previous caesareans  GROUP 5.2
  8.  Singleton cephalic, term, nulliparous, no previous
      caesarean, spontaneous labour ............................ GROUP 1
  9.  ... same, induced labour ................................. GROUP 2a
  10. ... same, pre-labour caesarean ........................... GROUP 2b
  11. Singleton cephalic, term, multiparous, no previous
      caesarean, spontaneous labour ............................ GROUP 3
  12. ... same, induced labour ................................. GROUP 4a
  13. ... same, pre-labour caesarean ........................... GROUP 4b

THE ORDER MATTERS. Groups 6 to 10 all include women with a previous
caesarean, which is why the previous-caesarean question comes after them.
A multiparous breech with a previous caesarean is group 7, not group 5.

If a record makes you hesitate, say so in YOUR_NOTES. The hesitation is
data: it means the definition is ambiguous against this data source, and
that belongs in the limitations section.
"""


if __name__ == "__main__":
    if "--score" in sys.argv:
        score()
    else:
        build()

---

## Summary

### SHAP findings
| Finding | Implication |
|---------|------------|
| Anthropometric factors dominate (~77% overall) | The model is driven by body size and reproductive history |
| Medical risk factors contribute ~6% overall | Documented indications explain very little of who gets a CS |
| `PRIORLIVE` inflated inside groups by mean\|SHAP\| | **Corrected:** switched to std(SHAP) for within-group ranking — PRIORLIVE dropped to near-zero in Groups 1 and 2a |
| Within-group ROC-AUC: **0.678–0.705** | The honest description of model skill (not the headline 0.7991) |

### Temporal validation
| What | Result |
|------|--------|
| Classifier stability | Group sizes and rates match across years ✓ |
| Model discrimination | ROC-AUC holds ✓ |
| Model calibration | Mean absolute error stable ✓ |
| Conclusion | The model transfers to unseen data from a different year |

### Kappa audit
| Item | Status |
|------|--------|
| Tool | Ready ✓ |
| 50-record blinded worksheet | Can be generated ✓ |
| Manual classification | Not yet executed (stated limitation) |
| Existing validation | 21 tests + external rates + temporal ✓ |

### Next notebooks
- **Notebook 6** → Export & packaging
- **00_MASTER.ipynb** → Results-only notebook for the committee

---
*End of Notebook 5*